In [ ]:
# ==========================================
# MULE ACCOUNT DETECTION
# PHASE 3 : FEATURE EXPLORATION
# & FEATURE DISCOVERY
# ==========================================

import pandas as pd
import numpy as np

from sklearn.feature_selection import mutual_info_classif

# ------------------------------------------
# LOAD PREPROCESSED DATASET
# ------------------------------------------

df = pd.read_csv(
    "../data/processed/preprocessed_data.csv"
)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)

# ------------------------------------------
# TARGET COLUMN
# ------------------------------------------

target_col = "F3924"

# ------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------

print("\nTARGET DISTRIBUTION")
print("-" * 40)

print(df[target_col].value_counts())

# ------------------------------------------
# BANK PROVIDED FEATURES
# ------------------------------------------

bank_features = [
    "F115","F321","F527","F531","F670",
    "F1692","F2082","F2122","F2582",
    "F2678","F2737","F2956","F3043",
    "F3836","F3887","F3889",
    "F3891","F3894"
]

# ------------------------------------------
# BANK FEATURE ANALYSIS
# ------------------------------------------

print("\n")
print("=" * 60)
print("BANK FEATURE ANALYSIS")
print("=" * 60)

for feature in bank_features:

    if feature not in df.columns:
        continue

    print("\n" + "-" * 50)
    print("Feature :", feature)

    # NUMERICAL FEATURE
    if pd.api.types.is_numeric_dtype(df[feature]):

        result = (
            df.groupby(target_col)[feature]
            .agg(["mean","median","std"])
        )

        print(result)

    # CATEGORICAL FEATURE
    else:

        result = pd.crosstab(
            df[feature],
            df[target_col]
        )

        print(result)

# ------------------------------------------
# PREPARE FEATURES
# ------------------------------------------

print("\n")
print("=" * 60)
print("PREPARING FEATURES")
print("=" * 60)

X = df.drop(columns=[target_col])

y = df[target_col]

# ------------------------------------------
# ENCODE CATEGORICAL FEATURES
# ------------------------------------------

categorical_cols = X.select_dtypes(
    include=["object","string"]
).columns

for col in categorical_cols:

    X[col] = X[col].astype("category").cat.codes

print("Categorical Features Encoded")

# ------------------------------------------
# CORRELATION ANALYSIS
# ------------------------------------------

print("\n")
print("=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)

corr_df = pd.concat(
    [X, y],
    axis=1
)

correlations = (
    corr_df.corr()[target_col]
    .abs()
    .sort_values(ascending=False)
)

correlations = correlations.drop(target_col)

top_corr = correlations.head(50)

print("\nTOP 20 CORRELATED FEATURES")

print(top_corr.head(20))

# ------------------------------------------
# SAVE CORRELATION REPORT
# ------------------------------------------

top_corr.to_csv(
    "../reports/top_correlated_features.csv"
)

# ------------------------------------------
# MUTUAL INFORMATION
# ------------------------------------------

print("\n")
print("=" * 60)
print("MUTUAL INFORMATION ANALYSIS")
print("=" * 60)

mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    by="MI_Score",
    ascending=False
)

print("\nTOP 20 MUTUAL INFORMATION FEATURES")

print(mi_df.head(20))

# ------------------------------------------
# SAVE MI REPORT
# ------------------------------------------

mi_df.to_csv(
    "../reports/mutual_information_features.csv",
    index=False
)

# ------------------------------------------
# COMBINED FEATURE DISCOVERY
# ------------------------------------------

print("\n")
print("=" * 60)
print("FEATURE DISCOVERY SUMMARY")
print("=" * 60)

top_corr_features = set(
    top_corr.head(50).index
)

top_mi_features = set(
    mi_df.head(50)["Feature"]
)

combined_features = list(
    top_corr_features.union(
        top_mi_features
    )
)

print("\nTop Features Discovered")

print("Total Features:", len(combined_features))

for feature in sorted(combined_features):
    print(feature)

# ------------------------------------------
# SAVE DISCOVERED FEATURES
# ------------------------------------------

pd.DataFrame({
    "Feature": combined_features
}).to_csv(
    "../reports/discovered_features.csv",
    index=False
)

# ------------------------------------------
# FINAL SUMMARY
# ------------------------------------------

print("\n" + "=" * 60)

print("FEATURE EXPLORATION COMPLETE")

print("=" * 60)

print(
    "Total Original Features:",
    X.shape[1]
)

print(
    "Top Correlation Features:",
    len(top_corr_features)
)

print(
    "Top MI Features:",
    len(top_mi_features)
)

print(
    "Combined Discovered Features:",
    len(combined_features)
)

print("=" * 60)

In our dataset, **F3924 is the target (label) column**, while all the other columns such as F1, F2, F115, F200, etc. are **features (input variables)**.

The dataset is anonymized, which means the actual meanings of the feature columns are hidden for privacy reasons. Therefore, we do not know what F115, F200, or any other feature specifically represents. We only know that they contain numerical values that may help distinguish fraudulent transactions from normal transactions.

During feature analysis, each feature is examined separately. For example, for feature F115, the dataset is divided into two groups based on the target column F3924:

* Rows where F3924 = 0 (normal transactions)
* Rows where F3924 = 1 (fraudulent transactions)

The mean, median, and standard deviation of F115 are then calculated for each group.

Mean represents the average value of the feature.
Median represents the middle value after sorting the feature values.
Standard deviation represents how spread out the values are around the mean.

If the statistical values of a feature differ significantly between the fraud and non-fraud groups, that feature may contain useful information for fraud detection.

For example:

Normal transactions (F3924 = 0):
Mean(F115) = 0.588

Fraud transactions (F3924 = 1):
Mean(F115) = 0.721

Since the average value of F115 is higher for fraudulent transactions, F115 may be useful for distinguishing fraud from non-fraud cases.

Thus, feature selection does not require knowledge of the actual meaning of a feature. It relies on identifying whether the numerical values of a feature show different patterns for different target classes. Features that show stronger differences are considered more informative and are selected for model training.


In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [7]:
# ==========================================
# SAVE ALL DISCOVERED FEATURES TO TXT FILE
# ==========================================

feature_list = sorted(combined_features)

with open("all_discovered_features.txt", "w") as file:

    file.write("ALL DISCOVERED FEATURES\n")
    file.write("=" * 50 + "\n\n")

    for i, feature in enumerate(feature_list, start=1):
        file.write(f"{i}. {feature}\n")

    file.write("\n")
    file.write("=" * 50 + "\n")
    file.write(f"Total Features: {len(feature_list)}\n")

print("File Created Successfully!")
print("Saved as: all_discovered_features.txt")

File Created Successfully!
Saved as: all_discovered_features.txt


Feature discovery was performed using Correlation Analysis and Mutual Information Analysis on all 3015 available features. The process identified 95 candidate features that showed strong relationships with the target variable (F3924). Interestingly, none of the 18 bank-provided reference features appeared in the final discovered feature set, indicating that the dataset contains additional hidden patterns that may be more predictive of mule account activity. These 95 newly discovered features will be further analyzed and reduced during the feature selection phase to obtain the most informative subset for model development.